In [2]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV, TimeSeriesSplit
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from random import randint, uniform


In [3]:
df = pd.read_csv(r"C:\Users\User\Desktop\ML\Data\processed\maharashtra_model_ready.csv")
df.head(20)

,district,year,month,rainfall_mm,SoilMoi_0_10,SoilMoi_10_40,SoilMoi_40_100,SoilMoi_100_200,et_mean_mm,ndvi,...,et_mean_mm_lag1,et_mean_mm_lag2,et_mean_mm_lag3,rainfall_roll3,rainfall_roll6,month_sin,month_cos,target_1m_ahead,target_2m_ahead,target_3m_ahead
0,Ahmednagar,2015,6,127.566858,0.297236,0.297696,0.257036,0.349434,6.980997,0.319885,...,1.985492,1.919723,3.045763,51.386090,27.008090,1.224647e-16,-1.000000e+00,No Drought,No Drought,No Drought
1,Ahmednagar,2015,7,52.172304,0.234076,0.321313,0.282012,0.352772,10.912704,0.362202,...,6.980997,1.985492,1.919723,67.984222,35.689121,-5.000000e-01,-8.660254e-01,No Drought,No Drought,Moderate
2,Ahmednagar,2015,8,65.907127,0.283227,0.327283,0.296945,0.355740,15.930731,0.387364,...,10.912704,6.980997,1.985492,81.882096,46.662365,-8.660254e-01,-5.000000e-01,No Drought,Moderate,Extreme
3,Ahmednagar,2015,9,211.699473,0.343166,0.356753,0.334319,0.365064,15.043684,0.447875,...,15.930731,10.912704,6.980997,109.926301,80.656196,-1.000000e+00,-1.836970e-16,Moderate,Extreme,Extreme
4,Ahmednagar,2015,10,72.067995,0.289113,0.352400,0.357233,0.376176,9.686445,0.481645,...,15.043684,15.930731,10.912704,116.558199,92.271210,-8.660254e-01,5.000000e-01,Extreme,Extreme,Extreme
5,Ahmednagar,2015,11,28.747276,0.231718,0.315782,0.322258,0.364618,7.825525,0.432876,...,9.686445,15.043684,15.930731,104.171582,93.026839,-5.000000e-01,8.660254e-01,Extreme,Extreme,Extreme
6,Ahmednagar,2015,12,0.045454,0.198771,0.288359,0.292413,0.357824,4.214328,0.397299,...,7.825525,9.686445,15.043684,33.620242,71.773272,-2.449294e-16,1.000000e+00,Extreme,Extreme,No Drought
7,Ahmednagar,2016,1,0.004455,0.159775,0.263094,0.267861,0.353344,2.802606,0.340517,...,4.214328,7.825525,9.686445,9.599062,63.078630,5.000000e-01,8.660254e-01,Extreme,No Drought,No Drought
8,Ahmednagar,2016,2,1.242280,0.152059,0.248939,0.253355,0.350385,2.399675,0.300846,...,2.802606,4.214328,7.825525,0.430730,52.301156,8.660254e-01,5.000000e-01,No Drought,No Drought,No Drought
9,Ahmednagar,2016,3,3.732513,0.161982,0.242790,0.246408,0.348261,1.019703,0.272327,...,2.399675,2.802606,4.214328,1.659749,17.639996,1.000000e+00,6.123234e-17,No Drought,No Drought,No Drought


In [4]:
df = df.drop(df.filter(like="Unnamed:").columns, axis=1)

In [5]:
target_columns = ['target_1m_ahead', 'target_2m_ahead', 'target_3m_ahead']

In [6]:
features_exclude = ['future_drought_class', 'target_1m_ahead', 'target_2m_ahead', 'target_3m_ahead', 'month', 'date']
feature_columns = [col for col in df.columns if col not in features_exclude]

X = df[feature_columns]

In [7]:
target = 'target_1m_ahead'
y = df[target]

In [8]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder

In [9]:
#One hot encoding for districts
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')

X_ohe = pd.DataFrame(
    ohe.fit_transform(X[['district']]),
    columns=ohe.get_feature_names_out(['district']),
    index=X.index
)

# replace district with OHE
X = pd.concat([X.drop(columns=['district']), X_ohe], axis=1)

In [10]:
train_year = [2015, 2016, 2017, 2018, 2019, 2020, 2021]
test_year = [2022, 2023, 2024]

train_mask = df['year'].isin(train_year)
test_mask = df['year'].isin(test_year)

In [11]:
X_train = X[train_mask]
X_test = X[test_mask]

y_1m_train = y[train_mask]
y_1m_test = y[test_mask]

In [12]:
print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")
print("\nClass distribution in Training set:")
print(y_1m_train.value_counts())
print("\nClass distribution in Test set:")
print(y_1m_test.value_counts())

Training set shape: (2765, 69)
Test set shape: (1050, 69)

Class distribution in Training set:
target_1m_ahead
No Drought    1610
Extreme        877
Moderate       278
Name: count, dtype: int64

Class distribution in Test set:
target_1m_ahead
No Drought    732
Extreme       236
Moderate       82
Name: count, dtype: int64


In [13]:
#computing class weights
from sklearn.utils import class_weight
class_weights = class_weight.compute_class_weight('balanced',
                                                 classes=np.unique(y_1m_train),
                                                 y=y_1m_train)
class_weight_dict = dict(zip(np.unique(y_1m_train), class_weights))
print("\nClass Weights:", class_weight_dict)

# Create sample weights for the training data
sample_weights = y_1m_train.map(class_weight_dict)


Class Weights: {'Extreme': np.float64(1.0509312048650703), 'Moderate': np.float64(3.315347721822542), 'No Drought': np.float64(0.572463768115942)}


In [14]:
# Label encoding target variable
le = LabelEncoder()
y_1m_train_encoded = le.fit_transform(y_1m_train)

In [15]:
from xgboost import XGBClassifier
model_1m_xgb = XGBClassifier(random_state=42,
                         eval_metric='mlogloss',
                         use_label_encoder=False)

model_1m_xgb.fit(X_train, y_1m_train_encoded,
             sample_weight=sample_weights)

c:\Users\User\Desktop\ML\menv\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:42:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'mlogloss'


In [16]:
y_1m_test_encoded = le.transform(y_1m_test)

In [17]:
# Model testing
from sklearn.metrics import classification_report, confusion_matrix

# predictions on the test set
y_1m_pred_encoded = model_1m_xgb.predict(X_test)

# Converting predictions back to original labels for interpretation
y_1m_pred = le.inverse_transform(y_1m_pred_encoded)
print("Classification Report for 1-Month Model:\n")
print(classification_report(y_1m_test, y_1m_pred))
xgb_f1 = f1_score(y_1m_test, y_1m_pred, average="macro")

Classification Report for 1-Month Model:

              precision    recall  f1-score   support

     Extreme       0.81      0.86      0.83       236
    Moderate       0.34      0.39      0.36        82
  No Drought       0.94      0.91      0.92       732

    accuracy                           0.85      1050
   macro avg       0.70      0.72      0.71      1050
weighted avg       0.86      0.85      0.86      1050



In [18]:
import joblib
joblib.dump(model_1m_xgb, "xgb_model_1m.pkl")
joblib.dump(le, "label_encoder_xgb.pkl")

print("Model and label encoder saved successfully!")

Model and label encoder saved successfully!


In [21]:
import joblib

# X_ohe = one-hot encoded districts
feature_columns = X.columns.tolist()  # all features including OHE
joblib.dump(feature_columns, "XGB_feature_order.pkl")

['XGB_feature_order.pkl']